# 📊 Análisis, Limpieza e Ingesta de Datos (FETRI ➔ ClickHouse)

Este notebook está diseñado para realizar el proceso completo de **Extracción, Transformación y Carga (ETL)** de los rankings oficiales de la FETRI. 

### Objetivos del Notebook:
1. **Explorar** visualmente la matriz de datos del Excel de Triatlón Masculino 2025.
2. **Limpiar y normalizar** los datos de los atletas (Deportistas) y los eventos (Pruebas).
3. **Transformar la matriz horizontal** del Excel en filas verticales individuales de **Resultados** (filtrando nulos y calculando el puesto real de carrera).
4. **Conectar e Ingestar** los datos limpios en la base de datos analítica **ClickHouse**.

## 🛠️ 1. Configuración e Importación de Librerías

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import clickhouse_connect

pd.set_option('display.max_columns', 15)
print("¡Librerías cargadas con éxito!")

## 📂 2. Carga y Exploración de Datos en Bruto

Cargamos la matriz del Excel de Triatlón Masculino 2025 para observar su estructura interna.

In [ ]:
excel_path = 'triatlon_masculino/trm_2025_limpio.xlsx'

# Cargar la primera hoja del Excel
df_raw = pd.read_excel(excel_path)
print(f"Dimensiones de la matriz de entrada: {df_raw.shape[0]} atletas x {df_raw.shape[1]} columnas")
df_raw.head(5)

## 🧹 3. Limpieza y Extracción de Deportistas (Dimensión Mutable)

Extraemos la información demográfica de los atletas (Nombre, Documento, Club, Grupo de Edad), eliminando duplicados y estandarizando textos a mayúsculas.

In [ ]:
# Extraer columnas del atleta y renombrar
df_deportistas = df_raw[['DOCUMENTO', 'NOMBRE', 'CLUB', 'GGEE']].copy()
df_deportistas.columns = ['documento', 'nombre', 'club', 'ggee']

# Limpieza de nulos en la clave primaria
df_deportistas = df_deportistas.dropna(subset=['documento'])
df_deportistas['documento'] = df_deportistas['documento'].astype(str).str.strip()

# Estandarización de textos (mayúsculas y limpieza de espacios)
df_deportistas['nombre'] = df_deportistas['nombre'].astype(str).str.upper().str.strip()
df_deportistas['club'] = df_deportistas['club'].fillna('SIN CLUB').astype(str).str.upper().str.strip()
df_deportistas['ggee'] = df_deportistas['ggee'].fillna('DESCONOCIDO').astype(str).str.upper().str.strip()

# Eliminar atletas duplicados (mantener el registro más reciente)
df_deportistas = df_deportistas.drop_duplicates(subset=['documento'], keep='first')

print(f"Total de atletas únicos limpios: {df_deportistas.shape[0]}")
df_deportistas.head(5)

## 📅 4. Extracción de Pruebas y Enriquecimiento con los JSON (Dimensión Estática)

Leemos el archivo `championships%3A2025.json` para extraer la fecha, el nivel de dificultad (`championship_level`) y las coordenadas geográficas (`lat`, `lng`) de cada prueba de la temporada 2025.

In [ ]:
json_path = 'championships%3A2025.json'

with open(json_path, 'r', encoding='utf-8') as f:
    championships_data = json.load(f)

# Mapear los campeonatos del JSON a una lista plana
pruebas_list = []
for item in championships_data:
    # 'c' contiene los datos del campeonato principal
    c = item.get('c', {})
    pruebas_list.append({
        'id': c.get('Id'),
        'season': 2025,
        'nombre': c.get('Name', '').strip(),
        'championship_level': c.get('ChampionshipLevel', 'NIVEL NO DEFINIDO'),
        'fecha': c.get('StartDate', '')[:10], # Solo YYYY-MM-DD
        'ciudad': c.get('City', 'DESCONOCIDA').strip(),
        'provincia': c.get('Province', 'DESCONOCIDA').strip(),
        'lat': float(c.get('Lat', 0.0)) if c.get('Lat') else 0.0,
        'lng': float(c.get('Lng', 0.0)) if c.get('Lng') else 0.0
    })

df_pruebas = pd.DataFrame(pruebas_list)
print(f"Total de pruebas encontradas en el calendario 2025: {df_pruebas.shape[0]}")
df_pruebas.head(5)

## 🔄 5. Normalización de Resultados (Tabla de Hechos Central)

Para modelar en ClickHouse, transformamos la matriz horizontal del Excel en un formato vertical (atleta-carrera). Además, aplicamos dos reglas críticas:
1. **Eliminar participaciones vacías (`NaN`):** Solo guardamos los registros donde el atleta realmente compitió.
2. **Calcular el puesto dinámico:** Ordenamos a los deportistas que corrieron la misma prueba de mayor a menor puntuación y les asignamos su `puesto` real en esa carrera (1º, 2º, 3º...).

In [ ]:
resultados_list = []

# Identificar las columnas del Excel que son nombres de carreras (descartando metadatos)
metadatos_cols = ['PUESTO', 'PUNTOS', 'NOMBRE', 'CLUB', 'NUMERO PRUEBAS', 'GGEE', 'VIGENTE', 'DOCUMENTO']
carreras_cols = [col for col in df_raw.columns if col not in metadatos_cols]

print(f"Procesando {len(carreras_cols)} columnas de carreras...")

for carrera in carreras_cols:
    # Buscar el ID de la prueba en nuestro DataFrame de pruebas cruzándolo por nombre
    prueba_match = df_pruebas[df_pruebas['nombre'].str.upper() == carrera.strip().upper()]
    
    if prueba_match.empty:
        # Si no coincide exactamente, buscamos una coincidencia parcial o creamos un ID genérico
        prueba_id = 999900 + len(df_pruebas)
    else:
        prueba_id = int(prueba_match.iloc[0]['id'])
        
    # Filtrar atletas que corrieron esta carrera (puntos no nulos y > 0)
    df_carrera = df_raw[['DOCUMENTO', carrera]].copy()
    df_carrera.columns = ['deportista_documento', 'puntos']
    df_carrera = df_carrera.dropna(subset=['puntos'])
    df_carrera = df_carrera[df_carrera['puntos'] > 0]
    
    if not df_carrera.empty:
        # Calcular el puesto real en la carrera ordenando por puntos de mayor a menor
        df_carrera = df_carrera.sort_values(by='puntos', ascending=False)
        df_carrera['puesto'] = range(1, len(df_carrera) + 1)
        
        # Añadir metadatos adicionales
        df_carrera['season'] = 2025
        df_carrera['ranking_type_id'] = 6 # 6 = Triatlón Masculino
        df_carrera['prueba_id'] = prueba_id
        
        resultados_list.append(df_carrera)

# Concatenar todos los dataframes de resultados
df_resultados = pd.concat(resultados_list, ignore_index=True)
df_resultados = df_resultados[['season', 'ranking_type_id', 'deportista_documento', 'prueba_id', 'puntos', 'puesto']]

print(f"Total de participaciones individuales limpias encontradas: {df_resultados.shape[0]}")
df_resultados.head(10)

## 📤 6. Conexión e Ingesta en la Base de Datos ClickHouse

Establecemos la conexión con tu contenedor de ClickHouse local en `localhost:8123` e insertamos las tres tablas principales ya limpias y estructuradas.

In [ ]:
# Configuración del cliente
client = clickhouse_connect.get_client(
    host='localhost',
    port=8123,
    username='admin',
    password='admin'
)
print("Conexión con ClickHouse establecida correctamente!")

# 1. Insertar Catálogo de Disciplina
df_cat = pd.DataFrame([
    {'id': 6, 'nombre': 'TRIATLÓN MASCULINO', 'siglas': 'TRM'}
])
client.insert('fetri.ranking_types', df_cat)
print("Catálogo ranking_types insertado!")

# 2. Insertar Pruebas
client.insert('fetri.pruebas', df_pruebas)
print(f"Insertadas {df_pruebas.shape[0]} pruebas en la tabla fetri.pruebas!")

# 3. Insertar Deportistas
# Añadir fecha_actualizacion para cumplir con el motor ReplacingMergeTree
df_deportistas['fecha_actualizacion'] = pd.Timestamp.now()
client.insert('fetri.deportistas', df_deportistas)
print(f"Insertados {df_deportistas.shape[0]} deportistas en la tabla fetri.deportistas!")

# 4. Insertar Resultados
client.insert('fetri.resultados', df_resultados)
print(f"Insertados {df_resultados.shape[0]} resultados individuales en la tabla fetri.resultados!")

print("\n🎉 ¡PROCESO ETL COMPLETADO CON ÉXITO! DATASET LISTO PARA EL MODELO PREDICTIVO.")

## 🔍 7. Consulta de Validación en ClickHouse

Hacemos una consulta rápida de prueba para verificar que todas las tablas están perfectamente relacionadas e insertadas.

In [ ]:
query = """
SELECT 
    d.nombre AS atleta,
    d.club,
    p.nombre AS carrera,
    p.championship_level AS dificultad,
    r.puntos,
    r.puesto AS puesto_carrera
FROM fetri.resultados r
LEFT JOIN fetri.deportistas d FINAL ON r.deportista_documento = d.documento
LEFT JOIN fetri.pruebas p ON r.prueba_id = p.id
WHERE r.puesto <= 3
LIMIT 10
"""

df_test = client.query_df(query)
df_test